# Task 3 — Customer Churn / Sales Trend Analysis

## E-commerce Sales Dataset

This project performs an end-to-end analysis of an e-commerce transaction dataset to identify revenue trends, customer segments, and potential churn/inactivity.

**Dataset:** UCI Online Retail (Chen, 2015), a real transactional dataset covering a UK-based online retailer from December 2010 to December 2011.

> **Important:** The UCI Online Retail dataset does not contain a direct `Churn` column. Therefore, this notebook creates an **inactivity-based churn proxy**: a customer is classified as "At Risk / Churned" when they have made no purchase during the final 90 days of the observed dataset period. This is an analytical definition, not a confirmed business churn label.


## Objectives

- Load and inspect a real e-commerce transaction dataset.
- Clean invalid transactions and missing customer records.
- Perform feature engineering and date extraction.
- Calculate revenue from quantity × unit price.
- Build monthly revenue trends and pivot tables.
- Identify top products, countries, and customer segments.
- Build customer-level RFM-style features: Recency, Frequency, and Monetary value.
- Create an inactivity-based churn proxy.
- Produce actionable business recommendations.


In [ ]:
# Install once if needed:
# !pip install ucimlrepo openpyxl pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("Libraries loaded successfully.")


## 1. Load the UCI Online Retail Dataset

In [ ]:
# UCI Online Retail dataset
# The ucimlrepo package retrieves the official dataset.
from ucimlrepo import fetch_ucirepo

online_retail = fetch_ucirepo(id=352)

# The transaction features are returned as a pandas DataFrame.
df = online_retail.data.features.copy()

print("Shape:", df.shape)
df.head()


In [ ]:
# If you downloaded Online Retail.xlsx manually instead, use:
# df = pd.read_excel("data/Online Retail.xlsx")

df.info()


## 2. Data Cleaning

The dataset contains cancellations and returns. For sales-revenue analysis, cancelled invoices and non-positive quantities/prices are excluded. Customer-level analysis is limited to records with a known CustomerID.


In [ ]:
# Convert dates and standardize types
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = df["CustomerID"].astype("Int64")

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Flag cancellations
df["IsCancellation"] = df["InvoiceNo"].astype(str).str.startswith("C")

# Keep valid positive sales for revenue analysis
sales = df[
    (~df["IsCancellation"]) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0) &
    (df["InvoiceDate"].notna())
].copy()

# Revenue feature
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]

print("Raw/cleaned transaction rows:", len(df))
print("Valid sales rows:", len(sales))
print("Known-customer sales rows:", sales["CustomerID"].notna().sum())
print("Total revenue:", round(sales["Revenue"].sum(), 2))


## 3. Feature Engineering & Date Extraction

In [ ]:
sales["Year"] = sales["InvoiceDate"].dt.year
sales["Month"] = sales["InvoiceDate"].dt.month
sales["MonthName"] = sales["InvoiceDate"].dt.strftime("%b")
sales["YearMonth"] = sales["InvoiceDate"].dt.to_period("M").astype(str)
sales["Quarter"] = sales["InvoiceDate"].dt.to_period("Q").astype(str)
sales["DayOfWeek"] = sales["InvoiceDate"].dt.day_name()

# Customer-level analysis
customer_sales = sales.dropna(subset=["CustomerID"]).copy()

print(customer_sales[[
    "InvoiceDate", "Year", "Month", "YearMonth",
    "Quarter", "DayOfWeek", "Revenue"
]].head())


## 4. Monthly Revenue Trend

In [ ]:
monthly_revenue = (
    sales.groupby("YearMonth", as_index=False)["Revenue"]
    .sum()
    .sort_values("YearMonth")
)

monthly_revenue["MoM_Growth_%"] = monthly_revenue["Revenue"].pct_change() * 100

display(monthly_revenue)

plt.figure(figsize=(12, 5))
plt.plot(monthly_revenue["YearMonth"], monthly_revenue["Revenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Interpretation:** The monthly trend shows how revenue changes over the observed period. Month-over-month growth highlights periods of acceleration or decline and can help identify seasonality.

## 5. Pivot Table — Revenue by Year and Month

In [ ]:
revenue_pivot = pd.pivot_table(
    sales,
    values="Revenue",
    index="Month",
    columns="Year",
    aggfunc="sum",
    fill_value=0
)

display(revenue_pivot)


## 6. Top Products by Revenue

In [ ]:
top_products = (
    sales.groupby("Description", as_index=False)
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum"),
        Orders=("InvoiceNo", "nunique")
    )
    .sort_values("Revenue", ascending=False)
    .head(10)
)

display(top_products)

plt.figure(figsize=(10, 6))
plot_data = top_products.sort_values("Revenue")
plt.barh(plot_data["Description"].str.slice(0, 35), plot_data["Revenue"])
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


## 7. Customer RFM-Style Segmentation

In [ ]:
analysis_date = sales["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    customer_sales.groupby("CustomerID")
    .agg(
        Recency=("InvoiceDate", lambda x: (analysis_date - x.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("Revenue", "sum")
    )
    .reset_index()
)

# Quantile scores: 1 = lower, 5 = higher.
rfm["R_Score"] = pd.qcut(rfm["Recency"].rank(method="first"), 5, labels=[5,4,3,2,1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)

rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

def segment(row):
    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "Champions"
    if row["R_Score"] >= 4 and row["F_Score"] >= 3:
        return "Loyal / Active"
    if row["R_Score"] <= 2 and row["F_Score"] >= 3:
        return "At Risk"
    if row["R_Score"] <= 2 and row["F_Score"] <= 2:
        return "Inactive"
    return "Potential"

rfm["Segment"] = rfm.apply(segment, axis=1)

display(rfm.head())
print(rfm["Segment"].value_counts())


## 8. Inactivity-Based Churn Proxy

In [ ]:
# A customer is considered potentially churned if their last purchase
# occurred more than 90 days before the end of the observed period.
CHURN_DAYS = 90

rfm["Churn_Proxy"] = np.where(
    rfm["Recency"] > CHURN_DAYS,
    "At Risk / Churned",
    "Active"
)

churn_summary = (
    rfm["Churn_Proxy"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Customers")
)

churn_summary["Percentage"] = (
    churn_summary["Customers"] / churn_summary["Customers"].sum() * 100
)

display(churn_summary)


**Interpretation:** This is a proxy for customer inactivity, not a verified churn label. A business could replace the 90-day threshold with its own customer-retention definition.

## 9. Revenue by Customer Segment

In [ ]:
segment_summary = (
    rfm.groupby("Segment", as_index=False)
    .agg(
        Customers=("CustomerID", "count"),
        Revenue=("Monetary", "sum"),
        Avg_Revenue_Per_Customer=("Monetary", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

display(segment_summary)

plt.figure(figsize=(9, 5))
plt.bar(segment_summary["Segment"], segment_summary["Revenue"])
plt.title("Revenue by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


## 10. Revenue by Country

In [ ]:
country_summary = (
    sales.groupby("Country", as_index=False)["Revenue"]
    .sum()
    .sort_values("Revenue", ascending=False)
    .head(10)
)

display(country_summary)

plt.figure(figsize=(10, 5))
plt.bar(country_summary["Country"], country_summary["Revenue"])
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Country")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 11. Business Recommendations

In [ ]:
# Generate a few data-driven summary metrics for the final discussion.
top_segment = segment_summary.iloc[0]
top_country = country_summary.iloc[0]
at_risk = int((rfm["Churn_Proxy"] == "At Risk / Churned").sum())
total_customers = len(rfm)

print("BUSINESS SUMMARY")
print("----------------")
print(f"Top revenue customer segment: {top_segment['Segment']}")
print(f"Revenue from top segment: £{top_segment['Revenue']:,.2f}")
print(f"Top revenue country: {top_country['Country']}")
print(f"Top country revenue: £{top_country['Revenue']:,.2f}")
print(f"Customers flagged by 90-day inactivity proxy: {at_risk:,} of {total_customers:,}")


### Recommended actions

1. **Protect high-value active customers:** provide loyalty benefits, early access, or personalized product offers to Champions and Loyal/Active customers.
2. **Win back inactive customers:** create targeted reactivation campaigns for customers above the 90-day inactivity threshold.
3. **Increase repeat purchases:** use product recommendations and cross-selling based on customers' previous purchases.
4. **Monitor monthly trends:** investigate months with significant month-over-month declines and plan promotions around seasonal demand.
5. **Focus on high-revenue markets:** prioritize inventory, campaigns, and localized offers in the strongest revenue-generating countries.
6. **Track the churn definition:** validate the 90-day inactivity proxy against actual business retention data before using it as an official churn KPI.


## Final Conclusion

This analysis demonstrates an end-to-end business intelligence workflow: data preparation, date extraction, revenue calculation, pivot-table analysis, trend analysis, customer segmentation, inactivity-based churn analysis, and business recommendations.

The project uses the public UCI Online Retail dataset. Because the dataset contains transactions rather than a direct churn label, churn is represented transparently as a 90-day inactivity proxy. This makes the analysis reproducible while leaving room for a business-specific churn definition.


## Dataset Source

Chen, D. (2015). **Online Retail**. UCI Machine Learning Repository. DOI: 10.24432/C5BW33.

The dataset contains 541,909 transactions from a UK-based online retailer between 1 December 2010 and 9 December 2011.
